# Optimizer comparison on classification using Digits dataset and Logistic Regression

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda, Compose

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import Logistic
from src.utils import data_load_and_prep, build_sampler
from src.training import train
from src.newton_optimizer import NewtonOptimizer

EXPERIMENT_NAME = "optimizer-comparison-logreg-MNIST"

In [2]:
# Loading MNIST dataset
transform = Compose([
    ToTensor(),
    Lambda(lambda x: x.view(-1)),
])
train_data = datasets.MNIST(root="../data", train=True, download=False, transform=transform)
test_data = datasets.MNIST(root="../data", train=False, download=False, transform=transform)

In [3]:
train_loader = DataLoader(train_data, batch_size=len(train_data), shuffle=True)
test_loader = DataLoader(test_data, batch_size=len(test_data), shuffle=False)

## Adam

In [29]:
loss_fn = nn.CrossEntropyLoss()
adam_model = Logistic(input_dim=784, output_dim=10)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Adam optimizaer

In [30]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=50, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/30 15:58:31 INFO mlflow.tracking.fluent: Experiment with name 'optimizer-comparison-logreg-MNIST' does not exist. Creating a new experiment.


Epoch 000 | train_loss=1.8959 | train_acc=0.537 | test_loss=1.8903 | test_acc=0.541 | 
Epoch 005 | train_loss=0.8219 | train_acc=0.805 | test_loss=0.7995 | test_acc=0.812 | 
Epoch 010 | train_loss=0.5631 | train_acc=0.844 | test_loss=0.5405 | test_acc=0.852 | 
Epoch 015 | train_loss=0.4693 | train_acc=0.866 | test_loss=0.4495 | test_acc=0.874 | 
Epoch 020 | train_loss=0.4211 | train_acc=0.879 | test_loss=0.4020 | test_acc=0.887 | 
Epoch 025 | train_loss=0.3909 | train_acc=0.887 | test_loss=0.3738 | test_acc=0.894 | 
Epoch 030 | train_loss=0.3702 | train_acc=0.894 | test_loss=0.3557 | test_acc=0.899 | 
Epoch 035 | train_loss=0.3546 | train_acc=0.899 | test_loss=0.3423 | test_acc=0.903 | 
Epoch 040 | train_loss=0.3426 | train_acc=0.903 | test_loss=0.3319 | test_acc=0.905 | 
Epoch 045 | train_loss=0.3330 | train_acc=0.906 | test_loss=0.3238 | test_acc=0.909 | 


2026/03/30 16:05:33 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=0.3266 | train_acc=0.908 | test_loss=0.3183 | test_acc=0.911 | 


## l-BFGS optimizer

In [31]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = Logistic(784, 10)
classical_device = "cuda" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              line_search_fn=None,
                              )

In [32]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="lbfgs-optimizer"
)

Epoch 000 | train_loss=2.3478 | train_acc=0.083 | test_loss=2.3460 | test_acc=0.082 | 
Epoch 005 | train_loss=2.1417 | train_acc=0.412 | test_loss=2.1359 | test_acc=0.428 | 
Epoch 010 | train_loss=1.9531 | train_acc=0.632 | test_loss=1.9440 | test_acc=0.635 | 
Epoch 015 | train_loss=1.7902 | train_acc=0.741 | test_loss=1.7781 | test_acc=0.749 | 
Epoch 020 | train_loss=1.6591 | train_acc=0.794 | test_loss=1.6450 | test_acc=0.801 | 
Epoch 025 | train_loss=1.5447 | train_acc=0.817 | test_loss=1.5289 | test_acc=0.828 | 
Epoch 030 | train_loss=1.4523 | train_acc=0.834 | test_loss=1.4352 | test_acc=0.844 | 
Epoch 035 | train_loss=1.3726 | train_acc=0.844 | test_loss=1.3545 | test_acc=0.854 | 
Epoch 040 | train_loss=1.3021 | train_acc=0.852 | test_loss=1.2833 | test_acc=0.859 | 
Epoch 045 | train_loss=1.2378 | train_acc=0.858 | test_loss=1.2186 | test_acc=0.865 | 


2026/03/30 16:12:39 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=1.1909 | train_acc=0.862 | test_loss=1.1714 | test_acc=0.870 | 


## Newton-Raphson optimizer

In [33]:
loss_fn = nn.CrossEntropyLoss()
newton_model = Logistic(784, 10)
classical_device = "cuda" 
newton_optimizer = NewtonOptimizer(newton_model.parameters(), 
                              lr=0.1,
                              max_iter=1,
                              damping=1e-4,
                              )

In [34]:
train(
    model=newton_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=newton_optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="newton-optimizer"
)

Epoch 000 | train_loss=1.8109 | train_acc=0.764 | test_loss=1.8087 | test_acc=0.767 | 
Epoch 005 | train_loss=1.0036 | train_acc=0.889 | test_loss=0.9979 | test_acc=0.889 | 
Epoch 010 | train_loss=0.6886 | train_acc=0.904 | test_loss=0.6855 | test_acc=0.903 | 
Epoch 015 | train_loss=0.5149 | train_acc=0.913 | test_loss=0.5157 | test_acc=0.909 | 
Epoch 020 | train_loss=0.4089 | train_acc=0.920 | test_loss=0.4143 | test_acc=0.914 | 
Epoch 025 | train_loss=0.3413 | train_acc=0.925 | test_loss=0.3515 | test_acc=0.919 | 
Epoch 030 | train_loss=0.2974 | train_acc=0.929 | test_loss=0.3127 | test_acc=0.921 | 
Epoch 035 | train_loss=0.2687 | train_acc=0.932 | test_loss=0.2895 | test_acc=0.922 | 
Epoch 040 | train_loss=0.2501 | train_acc=0.935 | test_loss=0.2765 | test_acc=0.924 | 
Epoch 045 | train_loss=0.2384 | train_acc=0.936 | test_loss=0.2702 | test_acc=0.926 | 


2026/03/30 16:24:18 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=0.2323 | train_acc=0.937 | test_loss=0.2682 | test_acc=0.926 | 


## Quadriatic Annealing optimizer (cpu)

In [4]:
loss_fn = nn.CrossEntropyLoss()
model = Logistic(784, 10)
classical_device = "cpu"

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

In [5]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cpu-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Epoch 000 | train_loss=2.2476 | train_acc=0.173 | test_loss=2.2493 | test_acc=0.177 | 
Epoch 005 | train_loss=2.1166 | train_acc=0.311 | test_loss=2.1189 | test_acc=0.309 | 
Epoch 010 | train_loss=2.0052 | train_acc=0.406 | test_loss=2.0079 | test_acc=0.409 | 
Epoch 015 | train_loss=1.9040 | train_acc=0.482 | test_loss=1.9072 | test_acc=0.486 | 
Epoch 020 | train_loss=1.8107 | train_acc=0.523 | test_loss=1.8114 | test_acc=0.528 | 
Epoch 025 | train_loss=1.7268 | train_acc=0.543 | test_loss=1.7258 | test_acc=0.549 | 
Epoch 030 | train_loss=1.6452 | train_acc=0.589 | test_loss=1.6429 | test_acc=0.593 | 
Epoch 035 | train_loss=1.5738 | train_acc=0.604 | test_loss=1.5720 | test_acc=0.605 | 
Epoch 040 | train_loss=1.5036 | train_acc=0.626 | test_loss=1.5001 | test_acc=0.625 | 
Epoch 045 | train_loss=1.4369 | train_acc=0.658 | test_loss=1.4320 | test_acc=0.662 | 


2026/04/02 09:03:45 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=1.3889 | train_acc=0.671 | test_loss=1.3826 | test_acc=0.676 | 


## Quadratic Annealing optimizer (cpu + momentum)

In [36]:
loss_fn = nn.CrossEntropyLoss()
model = Logistic(784, 10)
classical_device = "cpu"

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.1,
    num_reads=100,
    beta=0.9,
)

In [37]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cpu-momentum-optimizer"
)

Epoch 000 | train_loss=2.3002 | train_acc=0.166 | test_loss=2.2934 | test_acc=0.172 | 
Epoch 005 | train_loss=2.0081 | train_acc=0.347 | test_loss=2.0047 | test_acc=0.352 | 
Epoch 010 | train_loss=1.7768 | train_acc=0.502 | test_loss=1.7743 | test_acc=0.498 | 
Epoch 015 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 020 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 025 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 030 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 035 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 040 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 045 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 


2026/03/30 16:31:50 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
